In [1]:
# Imports
from google.colab import drive
import os
import scipy.sparse as sp
import pandas as pd
import scipy.stats as stats
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
import json
import joblib
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
from functools import partial
from hyperopt import space_eval
from sklearn.metrics import classification_report

In [2]:
# Mount drive, if needed
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Path to Google Drive project directory
drive_project_dir = '/content/drive/MyDrive/twitter-multimodal-hate-speech-classifier'

# Set project directory as current working directory
os.chdir(drive_project_dir)

# Set path to directory with data files in Google Drive
drive_data_dir = 'data' # relative path
os.makedirs(drive_data_dir, exist_ok=True)

# Set path to directory with trained models in Google Drive
drive_models_dir = 'models'
os.makedirs(drive_models_dir, exist_ok=True)

In [4]:
# Load preprocessed data and its target labels

# Preprocessing + TF-IDF vectorizaton: sparse
X_train_tfidf_processed_matrix = sp.load_npz(os.path.join(drive_data_dir, 'X_train_tfidf_processed_matrix.npz'))
X_val_tfidf_processed_matrix = sp.load_npz(os.path.join(drive_data_dir, 'X_val_tfidf_processed_matrix.npz'))

# Labels
y_train = pd.read_parquet(os.path.join(drive_data_dir, 'y_train.parquet'))['target']
y_val = pd.read_parquet(os.path.join(drive_data_dir, 'y_val.parquet'))['target']

In [6]:
# Define label map to encode numeric labels into strings
label_map = {0: 'NotHate', 1: 'Racist', 2: 'Sexist', 3: 'Homophobe', 4: 'Religion', 5: 'OtherHate'}

In [5]:
# Compute sample weights for current imbalanced multiclass problem
sample_weights = compute_sample_weight('balanced', y_train)

## TF-IDF + XGBoost: Random Search Hyperprarameter tuning


In [ ]:
# Define parameter space
param_space_tf_idf_xgb = {
# randint(low, high) == random integer in the range
'n_estimators': stats.randint(200, 1200),
'max_depth': stats.randint(3, 10),
'min_child_weight': stats.randint(1, 10),

# uniform(loc, scale) == range from 'loc' to 'loc + scale'
'colsample_bytree': stats.uniform(0.3, 0.6), # Range: 0.3 to 0.9
'subsample': stats.uniform(0.5, 0.5), # Range: 0.5 to 1.0

# loguniform(a, b) == range from 'a' to 'b' sampled logarithmically
'learning_rate': stats.loguniform(0.01, 0.2),

# Discrete values lists
'reg_alpha': [0, 1e-3, 0.01, 0.1, 1.0, 5.0, 10.0],
'reg_lambda': [0.1, 1.0, 2.0, 5.0, 10.0]
}

In [ ]:
# Setup base model
model_tf_idf_xgb_base = xgb.XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    early_stopping_rounds=25,
    tree_method="hist",
    device="cuda",
    random_state=42
)

In [ ]:
# Configure Random Search
random_search = RandomizedSearchCV(
    estimator=model_tf_idf_xgb_base,
    param_distributions=param_space_tf_idf_xgb,
    n_iter=30, # num of parameter combinations to try
    scoring='f1_macro',
    cv=3, # 3-fold cross-validation
    verbose=3,
    random_state=42,
    n_jobs=1 # must be 1 when using GPU
)

In [ ]:
# Execute Random Search
%%time
random_search.fit(
    X_train_tfidf_processed_matrix,
    y_train,
    eval_set=[(X_val_tfidf_processed_matrix, y_val)],
    sample_weight=sample_weights,
    verbose=False
)

# View the best results
print("Best Parameters: ", random_search.best_params_)
print("Best Macro F1: ", round(random_search.best_score_, 4))

Fitting 3 folds for each of 30 candidates, totalling 90 fits
[CV 1/3] END colsample_bytree=0.5247240713084175, learning_rate=0.17254716573280354, max_depth=5, min_child_weight=8, n_estimators=900, reg_alpha=1.0, reg_lambda=1.0, subsample=0.5779972601681014;, score=0.330 total time=   7.9s
[CV 2/3] END colsample_bytree=0.5247240713084175, learning_rate=0.17254716573280354, max_depth=5, min_child_weight=8, n_estimators=900, reg_alpha=1.0, reg_lambda=1.0, subsample=0.5779972601681014;, score=0.360 total time=   7.7s
[CV 3/3] END colsample_bytree=0.5247240713084175, learning_rate=0.17254716573280354, max_depth=5, min_child_weight=8, n_estimators=900, reg_alpha=1.0, reg_lambda=1.0, subsample=0.5779972601681014;, score=0.364 total time=  10.8s
[CV 1/3] END colsample_bytree=0.33485016730091965, learning_rate=0.13394334706750485, max_depth=6, min_child_weight=8, n_estimators=863, reg_alpha=0.01, reg_lambda=20.0, subsample=0.5282057895135501;, score=0.331 total time=  14.6s
[CV 2/3] END colsamp

In [ ]:
# Save best found settings for hyperparameters to file
file_path = os.path.join(drive_models_dir, 'random_search_best_params_tfidf_xgb.json')

with open(file_path, 'w') as f:
    json.dump(random_search.best_params_, f, indent=4)

In [ ]:
# Save trained model with best hyperprarameters
joblib.dump(random_search.best_estimator_, os.path.join(drive_models_dir, 'random_search_best_estimator_tfidf_xgb.joblib'))

['/content/drive/MyDrive/ML_course/Final-Project/models/model_xgb_tfidf_opt_hyperparams.joblib']

In [ ]:
# Make prediction using best configuration
train_pred_classes_randsearch_xgb = random_search.best_estimator_.predict(X_train_tfidf_processed_matrix)
val_pred_classes_randsearch_xgb = random_search.best_estimator_.predict(X_val_tfidf_processed_matrix)

In [ ]:
# Evaluate model's performance on train data
print(classification_report(y_train, train_pred_classes_randsearch_xgb, target_names=list(label_map.values())))

In [ ]:
# Evaluate model's performance on train data
print(classification_report(y_train, val_pred_classes_randsearch_xgb, target_names=list(label_map.values())))

In [ ]:
random_search.best_params_

{'colsample_bytree': np.float64(0.5297561248522739),
 'learning_rate': np.float64(0.18374967653799565),
 'max_depth': 5,
 'min_child_weight': 1,
 'n_estimators': 204,
 'reg_alpha': 0.001,
 'reg_lambda': 20.0,
 'subsample': np.float64(0.7468977981821954)}

In [ ]:
# Save the used Random Search instance
joblib.dump(random_search, os.path.join(drive_models_dir, 'random_search_instance_tfidf_xgb.joblib'))

['/content/drive/MyDrive/ML_course/Final-Project/models/random_search_instance_tf_idf_xgb.joblib']

## TF-IDF + XGBoost: Hyperopt Hyperprarameter tuning

In [ ]:
# Define objective function
def objective(params):

  # Copy params (shalow) to cast int so original 'params' argument stays untouched
  local_params = params.copy()

  # Cast int parameters
  if "n_estimators" in local_params:
    local_params["n_estimators"] = int(local_params["n_estimators"])
  if "max_depth" in local_params:
    local_params["max_depth"] = int(local_params["max_depth"])
  if "min_child_weight" in local_params:
    local_params["min_child_weight"] = int(local_params["min_child_weight"])

  # 3-Fold Stratified cross-validation
  skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
  cv_scores = []

  for train_idx, val_idx in skf.split(X_train_tfidf_processed_matrix, y_train):
    X_tr = X_train_tfidf_processed_matrix[train_idx]
    X_va = X_train_tfidf_processed_matrix[val_idx]
    y_tr = (y_train.iloc[train_idx] if hasattr(y_train, "iloc") else y_train[train_idx]) # handle both cases: matrix and dataframe
    y_va = (y_train.iloc[val_idx] if hasattr(y_train, "iloc") else y_train[val_idx])

    # Slice sample weights for the current training fold
    w_tr = sample_weights[train_idx]

    # Create XGBoost instance
    clf = xgb.XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        early_stopping_rounds=20,
        device="cuda",
        tree_method="hist",
        **local_params,
    )

    # Fit with fold-specific sample weights
    clf.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_va, y_va)], verbose=False)

    # Predict classes and calculate Macro F1-score for multiclass evaluation
    preds = clf.predict(X_va)
    score = f1_score(y_va, preds, average="macro")
    cv_scores.append(score)

  mean_macro_f1 = np.mean(cv_scores)

  return {"loss": -mean_macro_f1, "status": STATUS_OK} # minimization of loss -> negative macro F1

In [ ]:
# Define parameter space (for a quick search)
param_space_tf_idf_xgb = {
    "n_estimators": scope.int(hp.quniform("n_estimators", 100, 400, 1)),
    "max_depth": scope.int(hp.quniform("max_depth", 3, 8, 1)),
    "min_child_weight": scope.int(hp.quniform("min_child_weight", 1, 8, 1)),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.4, 0.8),
    "subsample": hp.uniform("subsample", 0.6, 1.0),
    "learning_rate": hp.loguniform("learning_rate", np.log(0.01), np.log(0.2)),
    "reg_alpha": hp.choice("reg_alpha", [0.1, 1.0, 5.0]),
    "reg_lambda": hp.choice("reg_lambda", [1.0, 2.0, 5.0])
}

In [ ]:
# Allow TPE to start optimization after 3 random trials since only 10 total evals (very quick search)
algo = partial(tpe.suggest, n_startup_jobs=3)

trials = Trials()

best_config = fmin(
    fn=objective,
    space=param_space_tf_idf_xgb,
    algo=algo,  # custom partial function
    max_evals=10,
    trials=trials,
    rstate=np.random.default_rng(42),
)

100%|██████████| 10/10 [17:35<00:00, 105.55s/trial, best loss: -0.43996131751554507]


In [ ]:
# Parameter conversion (for integers)
best_config['n_estimators'] = int(best_config['n_estimators'])
best_config['max_depth'] = int(best_config['max_depth'])
best_config['min_child_weight'] = int(best_config['min_child_weight'])

print("Best hyperparameter configuration: ", best_config)

Best hyperparameter configuration:  {'colsample_bytree': 0.7564597038310892, 'learning_rate': 0.11784461668874158, 'max_depth': 6, 'min_child_weight': 2, 'n_estimators': 285, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'subsample': 0.658785114914011}


In [ ]:
# Setup final model with optimized hyperparameters
final_xgboost = xgb.XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    verbosity=3,
    early_stopping_rounds=20,
    device="cuda",
    tree_method="hist",
    **best_config,
)

In [ ]:
# Save model instance with best hyperprarameters
joblib.dump(final_xgboost, os.path.join(drive_models_dir, 'hyperopt_best_tfidf_xgb_not_trained.joblib'))

['/content/drive/MyDrive/ML_course/Final-Project/models/model_xgb_tfidf_hyperopt.joblib']

In [ ]:
# Collect trials and best config together
hyperopt_results = {"trials": trials, "best_config": best_config}

# Save to disk
joblib.dump(hyperopt_results, os.path.join(drive_models_dir, 'hyperopt_results_tfidf_xgb.joblib'))

['/content/drive/MyDrive/ML_course/Final-Project/models/hyperopt_xgb_results.joblib']

In [ ]:
# Train model with best found hyperparameter settings
final_xgboost.fit(
    X_train_tfidf_processed_matrix,
    y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val_tfidf_processed_matrix, y_val)],
    verbose=False,
)

[14:19:42] ======== Monitor (0): HostSketchContainer ========
[14:19:42] AllReduce: 0.084244s, 1 calls @ 84244us

[14:19:42] MakeCuts: 0.099117s, 1 calls @ 99117us

[14:19:42] INFO: /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (125502, 10005, 2594841).
[14:19:42] INFO: /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (4197, 10005, 85722).
[14:19:42] DEBUG: /__w/xgboost/xgboost/src/gbm/gbtree.cc:126: Using tree method: 3
[14:19:42] DEBUG: /__w/xgboost/xgboost/src/tree/updater_gpu_hist.cu:766: [GPU Hist]: Configure
[14:19:42] INFO: /__w/xgboost/xgboost/src/data/ellpack_page.cu:180: Ellpack is sparse.
[14:19:42] ======== Monitor (0): ellpack_page ========
[14:19:42] CopyGHistToEllpack: 0.00699s, 1 calls @ 6990us

[14:19:42] InitCompressedData: 0.000195s, 1 calls @ 195us

[14:19:42] INFO: /__w/xgboost/xgboost/src/data/ellpack_page.cu:180: Ellpack is sparse.
[14:19:42] ======== Monito

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7564597038310892, device='cuda',
              early_stopping_rounds=20, enable_categorical=True,
              eval_metric='mlogloss', feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.11784461668874158,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=2, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=285, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
# Save trained model with best hyperprarameters to disk
joblib.dump(final_xgboost, os.path.join(drive_models_dir, 'hyperopt_best_tfidf_xgb_trained.joblib'))

['/content/drive/MyDrive/ML_course/Final-Project/models/model_xgb_tfidf_hyperopt_trained.joblib']

In [ ]:
# Evaluate best model
best_iter_range = (0, final_xgboost.best_iteration + 1)

train_final_preds_xgb = final_xgboost.predict(X_train_tfidf_processed_matrix, iteration_range=best_iter_range)
val_final_preds_xgb = final_xgboost.predict(X_val_tfidf_processed_matrix, iteration_range=best_iter_range)

In [ ]:
# Evaluate model's performance on train data
print(classification_report(y_train, train_final_preds_xgb, target_names=list(label_map.values())))

              precision    recall  f1-score   support

     NotHate       0.95      0.67      0.79    105346
      Racist       0.39      0.61      0.47      9503
      Sexist       0.17      0.95      0.29      2790
   Homophobe       0.39      1.00      0.56      3087
    Religion       0.35      1.00      0.52       131
   OtherHate       0.32      0.91      0.47      4645

    accuracy                           0.69    125502
   macro avg       0.43      0.85      0.52    125502
weighted avg       0.86      0.69      0.74    125502



In [ ]:
# Evaluate model's performance on train data
print(classification_report(y_val, val_final_preds_xgb, target_names=list(label_map.values())))

              precision    recall  f1-score   support

     NotHate       0.80      0.66      0.73      2500
      Racist       0.59      0.48      0.53       809
      Sexist       0.35      0.74      0.47       241
   Homophobe       0.61      0.95      0.75       252
    Religion       0.33      0.44      0.38         9
   OtherHate       0.57      0.80      0.66       386

    accuracy                           0.66      4197
   macro avg       0.54      0.68      0.59      4197
weighted avg       0.70      0.66      0.67      4197

